In [1]:
"""
Experimento fatorial: sensibilidade da análise probabilística de viabilidade
às escolhas de modelagem do analista.

Implementa o protocolo da Seção "Método" do artigo:
  - Amostragem por transformada inversa com números aleatórios comuns
    (Beckman & McKay, 1987) para isolar o efeito de cada escolha de
    modelagem do ruído de simulação.
  - Correlação induzida via cópula gaussiana (Iman & Conover, 1982).
  - Identidade Prop(VPL(i)<0) = F_TIR(i) para obter a curva de risco vs TMA
    sem resolver a TIR por iteração (Proposição "Equivalência dos critérios").
  - Validação: media e variância simuladas conferidas contra as formas
    fechadas das Proposições "Momentos do VPL" e "Momentos por família".

Sementes fixas -> reprodutível. Todas as tabelas usadas no capítulo de
Resultados são geradas por este script.
"""
import numpy as np
from scipy import stats
from dataclasses import dataclass

MASTER_SEED = 2026
N_DEFAULT = 100_000

# ---------------------------------------------------------------------------
# 1. Projeto canônico (Tabela "Projeto canônico e elicitação de três pontos")
# ---------------------------------------------------------------------------
CANON = {
    "I": dict(a=480_000, m=500_000, b=540_000),
    "R": dict(a=270_000, m=300_000, b=336_000),
    "C": dict(a=138_000, m=150_000, b=165_000),
}
N_ANOS = 5
TMA_BASE = 0.12


def fator_anuidade(tma, n=N_ANOS):
    """S = (1-(1+TMA)^-n)/TMA"""
    tma = np.asarray(tma, dtype=float)
    # limite quando tma -> 0
    return np.where(np.abs(tma) < 1e-12, float(n),
                     (1 - (1 + tma) ** (-n)) / tma)


# ---------------------------------------------------------------------------
# 2. Famílias de distribuição: mapeiam uniformes (0,1) -> valores, dado (a,m,b)
#    Todas recebem o MESMO vetor de uniformes -> números aleatórios comuns.
# ---------------------------------------------------------------------------
def ppf_triangular(u, a, m, b):
    c = (m - a) / (b - a)
    return stats.triang.ppf(u, c=c, loc=a, scale=b - a)


def ppf_pert(u, a, m, b, lam=4.0):
    alpha1 = 1 + lam * (m - a) / (b - a)
    alpha2 = 1 + lam * (b - m) / (b - a)
    return a + (b - a) * stats.beta.ppf(u, alpha1, alpha2)


def ppf_uniforme(u, a, m, b):
    # a moda é descartada por construção (Prop. "Momentos por família")
    return stats.uniform.ppf(u, loc=a, scale=b - a)


def ppf_normal_eq(u, a, m, b, ref="triangular"):
    """Normal casada nos dois primeiros momentos da família de referência
    (pareamento B: mesmos momentos, forma diferente)."""
    mu, var = MOMENTOS[ref](a, m, b)
    return stats.norm.ppf(u, loc=mu, scale=np.sqrt(var))


def momentos_triangular(a, m, b):
    mu = (a + m + b) / 3
    var = (a**2 + m**2 + b**2 - a*m - a*b - m*b) / 18
    return mu, var


def momentos_pert(a, m, b, lam=4.0):
    mu = (a + lam * m + b) / (lam + 2)
    var = (mu - a) * (b - mu) / (lam + 3)
    return mu, var


def momentos_uniforme(a, m, b):
    mu = (a + b) / 2
    var = (b - a) ** 2 / 12
    return mu, var


MOMENTOS = {
    "triangular": momentos_triangular,
    "pert": momentos_pert,
    "uniforme": momentos_uniforme,
}

FAMILIAS = {
    "Triangular": (ppf_triangular, "triangular"),
    "PERT": (ppf_pert, "pert"),
    "Uniforme": (ppf_uniforme, "uniforme"),
    "Normal eq.": (ppf_normal_eq, "triangular"),  # casada na triangular (pareamento B)
}


# ---------------------------------------------------------------------------
# 3. Núcleo: gera (I,R,C) sob uma família e correlação dadas, calcula o VPL
# ---------------------------------------------------------------------------
def gerar_variaveis(rng, elicit, familia, N, rho=0.0):
    """
    elicit: dict com chaves 'I','R','C' -> dict(a=,m=,b=)
    familia: nome em FAMILIAS
    rho: correlação (Pearson, cópula gaussiana) entre R e C; I é sempre
         independente (modelo canônico de incerteza de nível).
    Retorna I, R, C (arrays de tamanho N).
    """
    ppf_func, _ = FAMILIAS[familia]

    # --- I: independente, uniforme própria ---
    u_I = rng.random(N)
    I = ppf_func(u_I, **elicit["I"])

    # --- R, C: cópula gaussiana com correlação rho ---
    z1 = rng.standard_normal(N)
    z2_indep = rng.standard_normal(N)
    z2 = rho * z1 + np.sqrt(1 - rho**2) * z2_indep
    u_R = stats.norm.cdf(z1)
    u_C = stats.norm.cdf(z2)
    R = ppf_func(u_R, **elicit["R"])
    C = ppf_func(u_C, **elicit["C"])
    return I, R, C


def calcular_vpl(I, R, C, tma, n=N_ANOS):
    S = fator_anuidade(tma, n)
    return -I + (R - C) * S


def resumo(vpl, N):
    media = vpl.mean()
    dp = vpl.std(ddof=1)
    p_neg = (vpl < 0).mean()
    se_p = np.sqrt(p_neg * (1 - p_neg) / N)
    p5, p50, p95 = np.percentile(vpl, [5, 50, 95])
    return dict(media=media, dp=dp, p_neg=p_neg, se_p=se_p, p5=p5, p50=p50, p95=p95)


def momentos_exatos_vpl(elicit, familia_key, tma, rho=0.0, n=N_ANOS):
    """Proposição 'Momentos do VPL' combinada com 'Momentos por família'."""
    S = fator_anuidade(tma, n)
    mu_I, var_I = MOMENTOS[familia_key](**elicit["I"])
    mu_R, var_R = MOMENTOS[familia_key](**elicit["R"])
    mu_C, var_C = MOMENTOS[familia_key](**elicit["C"])
    sig_R, sig_C = np.sqrt(var_R), np.sqrt(var_C)
    media = -mu_I + (mu_R - mu_C) * S
    var = var_I + S**2 * (var_R + var_C - 2 * rho * sig_R * sig_C)
    return media, var


if __name__ == "__main__":
    print("Módulo carregado. Use os scripts eixo1.py .. eixo4.py para rodar.")

Módulo carregado. Use os scripts eixo1.py .. eixo4.py para rodar.


In [8]:
"""Eixo 1 — família de distribuição (pareamento A: mesmos a,m,b)."""

import numpy as np
from scipy.stats import norm

def rodar(N=N_DEFAULT, seed=MASTER_SEED):
    linhas = []
    for nome_familia, (_, ref_key) in FAMILIAS.items():
        rng = np.random.default_rng(seed)  # MESMA semente -> mesmos U por família
        I, R, C = gerar_variaveis(rng, CANON, nome_familia, N, rho=0.0)
        vpl = calcular_vpl(I, R, C, TMA_BASE)
        r = resumo(vpl, N)

        mu_exato, var_exato = momentos_exatos_vpl(CANON, ref_key, TMA_BASE, rho=0.0)
        sigma_exato = np.sqrt(var_exato)
        phi_pred = norm.cdf(-mu_exato / sigma_exato)
        delta = (r["p_neg"] - phi_pred) * 100

        # validação: media/DP simulados vs exatos (erro-padrao da media = dp/sqrt(N))
        se_media = r["dp"] / np.sqrt(N)
        z_media = (r["media"] - mu_exato) / se_media
        linhas.append(dict(
            familia=nome_familia, media=r["media"], dp=r["dp"],
            p_neg=r["p_neg"], se_p=r["se_p"],
            p5=r["p5"], p50=r["p50"], p95=r["p95"],
            mu_exato=mu_exato, sigma_exato=sigma_exato,
            phi_pred=phi_pred, delta_pp=delta, z_media=z_media,
        ))
    return linhas


if __name__ == "__main__":
    linhas = rodar()
    print(f"{'Família':<12}{'E[VPL] R$mil':>13}{'DP R$mil':>10}{'P(VPL<0)':>10}"
          f"{'+-SE(pp)':>9}{'Phi-pred':>10}{'delta(pp)':>10}{'z(media)':>9}")
    for l in linhas:
        print(f"{l['familia']:<12}"
              f"{l['media']/1000:>13.1f}"
              f"{l['dp']/1000:>10.1f}"
              f"{l['p_neg']*100:>9.2f}%"
              f"{l['se_p']*100:>8.2f}%"
              f"{l['phi_pred']*100:>9.1f}%"
              f"{l['delta_pp']:>+9.1f} "
              f"{l['z_media']:>8.2f}")
    print()
    print("P5 / P50 / P95 do VPL (R$ mil):")
    for l in linhas:
        print(f"  {l['familia']:<12} P5={l['p5']/1000:>7.1f}  "
              f"P50={l['p50']/1000:>7.1f}  P95={l['p95']/1000:>7.1f}")

Família      E[VPL] R$mil  DP R$mil  P(VPL<0) +-SE(pp)  Phi-pred delta(pp) z(media)
Triangular           37.7      54.1    25.43%    0.14%     24.3%     +1.1     0.52
PERT                 39.3      49.8    23.03%    0.13%     21.5%     +1.5     0.51
Uniforme             36.3      76.4    34.80%    0.15%     31.8%     +3.0     0.64
Normal eq.           37.7      54.1    24.29%    0.14%     24.3%     +0.0     0.49

P5 / P50 / P95 do VPL (R$ mil):
  Triangular   P5=  -50.2  P50=   36.7  P95=  129.0
  PERT         P5=  -41.4  P50=   38.5  P95=  122.3
  Uniforme     P5=  -86.3  P50=   36.0  P95=  159.7
  Normal eq.   P5=  -51.1  P50=   37.6  P95=  126.9


In [16]:
"""
EIXO 1 — PAREAMENTO B (mesmos momentos, forma diferente) — VERSÃO NOTEBOOK
Cole esta célula inteira e rode. Não depende de nenhuma outra célula.

O pareamento A (já rodado antes) dá às quatro famílias os MESMOS três
pontos elicitados (a,m,b) -- mas cada família implica uma variância
diferente a partir desses mesmos pontos (Corolário "Caso simétrico"). Logo
a diferença de risco do pareamento A mistura dois efeitos: (i) a variância
diferente que cada família produz, e (ii) a forma pura da distribuição.

O pareamento B isola (ii): calculamos a média e a variância EXATAS que a
PERT canônica produz para I, R e C, e então damos a cada família os
parâmetros (elicitação simétrica) que reproduzem exatamente essa mesma
média e variância. Com os momentos travados, qualquer diferença
remanescente em P(VPL<0) só pode vir da forma pura.
"""
import numpy as np
from scipy import stats
from scipy.stats import norm

MASTER_SEED = 2026
N_DEFAULT = 100_000

CANON = {
    "I": dict(a=480_000, m=500_000, b=540_000),
    "R": dict(a=270_000, m=300_000, b=336_000),
    "C": dict(a=138_000, m=150_000, b=165_000),
}
N_ANOS = 5
TMA_BASE = 0.12


def fator_anuidade(tma, n=N_ANOS):
    tma = np.asarray(tma, dtype=float)
    return np.where(np.abs(tma) < 1e-12, float(n),
                     (1 - (1 + tma) ** (-n)) / tma)


def ppf_triangular(u, a, m, b):
    c = (m - a) / (b - a)
    return stats.triang.ppf(u, c=c, loc=a, scale=b - a)


def ppf_pert(u, a, m, b, lam=4.0):
    alpha1 = 1 + lam * (m - a) / (b - a)
    alpha2 = 1 + lam * (b - m) / (b - a)
    return a + (b - a) * stats.beta.ppf(u, alpha1, alpha2)


def ppf_uniforme(u, a, m, b):
    return stats.uniform.ppf(u, loc=a, scale=b - a)


def momentos_pert(a, m, b, lam=4.0):
    mu = (a + lam * m + b) / (lam + 2)
    var = (mu - a) * (b - mu) / (lam + 3)
    return mu, var


def ppf_normal_eq(u, mu, var):
    return stats.norm.ppf(u, loc=mu, scale=np.sqrt(var))


def calcular_vpl(I, R, C, tma, n=N_ANOS):
    S = fator_anuidade(tma, n)
    return -I + (R - C) * S


def resumo(vpl, N):
    media = vpl.mean()
    dp = vpl.std(ddof=1)
    p_neg = (vpl < 0).mean()
    se_p = np.sqrt(p_neg * (1 - p_neg) / N)
    return dict(media=media, dp=dp, p_neg=p_neg, se_p=se_p)


# ---------------------------------------------------------------------------
# 1. Alvo de momentos: media e variancia exatas da PERT canonica
# ---------------------------------------------------------------------------
alvo = {}
for var, pontos in CANON.items():
    mu, v = momentos_pert(**pontos)
    alvo[var] = dict(mu=mu, var=v)

print("Alvo de momentos (media, variancia exatas da PERT canonica):")
for var, a in alvo.items():
    print(f"  {var}: mu={a['mu']:.1f}  var={a['var']:.1f}")
print()


def elicitacao_simetrica_para_alvo(fator_L):
    """Elicitacao simetrica (a,m,b) que reproduz exatamente (mu,var) do
    alvo, dado o fator L da familia (28=PERT, 24=Triangular, 12=Uniforme,
    Corolario 'Caso simetrico')."""
    elicit = {}
    for var, a in alvo.items():
        mu, v = a["mu"], a["var"]
        L = np.sqrt(fator_L * v)
        elicit[var] = dict(a=mu - L / 2, m=mu, b=mu + L / 2)
    return elicit


elicit_pert = elicitacao_simetrica_para_alvo(28)
elicit_triangular = elicitacao_simetrica_para_alvo(24)
elicit_uniforme = elicitacao_simetrica_para_alvo(12)

# ---------------------------------------------------------------------------
# 2. Simulacao de cada familia, com os momentos ja travados no alvo
# ---------------------------------------------------------------------------
configs = {
    "Triangular": (ppf_triangular, elicit_triangular),
    "PERT":       (ppf_pert,       elicit_pert),
    "Uniforme":   (ppf_uniforme,   elicit_uniforme),
}

linhas = []
for nome_familia, (ppf_func, elicit) in configs.items():
    rng = np.random.default_rng(MASTER_SEED)
    u_I = rng.random(N_DEFAULT)
    z1 = rng.standard_normal(N_DEFAULT)
    z2 = rng.standard_normal(N_DEFAULT)  # rho=0: isola so a forma
    u_R = norm.cdf(z1)
    u_C = norm.cdf(z2)
    I = ppf_func(u_I, **elicit["I"])
    R = ppf_func(u_R, **elicit["R"])
    C = ppf_func(u_C, **elicit["C"])
    vpl = calcular_vpl(I, R, C, TMA_BASE)
    r = resumo(vpl, N_DEFAULT)
    linhas.append(dict(familia=nome_familia, **r))

# "Normal eq.": usa diretamente (mu,var) do alvo -- ja reproduz os momentos
# exatos por construcao (referencia de forma "suporte irrestrito")
rng = np.random.default_rng(MASTER_SEED)
u_I = rng.random(N_DEFAULT)
z1 = rng.standard_normal(N_DEFAULT)
z2 = rng.standard_normal(N_DEFAULT)
u_R = norm.cdf(z1)
u_C = norm.cdf(z2)
I = ppf_normal_eq(u_I, alvo["I"]["mu"], alvo["I"]["var"])
R = ppf_normal_eq(u_R, alvo["R"]["mu"], alvo["R"]["var"])
C = ppf_normal_eq(u_C, alvo["C"]["mu"], alvo["C"]["var"])
vpl = calcular_vpl(I, R, C, TMA_BASE)
r = resumo(vpl, N_DEFAULT)
linhas.append(dict(familia="Normal eq.", **r))

# ---------------------------------------------------------------------------
# 3. Resultados
# ---------------------------------------------------------------------------
print(f"{'Família':<12}{'E[VPL] R$mil':>13}{'DP R$mil':>10}{'P(VPL<0)':>10}{'+-SE(pp)':>9}")
for l in linhas:
    print(f"{l['familia']:<12}"
          f"{l['media']/1000:>13.1f}"
          f"{l['dp']/1000:>10.1f}"
          f"{l['p_neg']*100:>9.2f}%"
          f"{l['se_p']*100:>8.2f}%")

base = next(l["p_neg"] for l in linhas if l["familia"] == "PERT")
print("\nDelta vs. PERT (pontos percentuais) -- efeito PURO de forma:")
for l in linhas:
    print(f"  {l['familia']:<12} {(l['p_neg']-base)*100:+.2f} p.p.")

# comparacao com o efeito TOTAL do pareamento A (numeros ja conhecidos)
efeito_total_A = 34.80 - 23.03  # Uniforme - PERT no pareamento A
efeito_forma_B = next(l["p_neg"] for l in linhas if l["familia"] == "Uniforme")*100 - base*100
print(f"\nEfeito total (pareamento A, Uniforme-PERT): {efeito_total_A:.2f} p.p.")
print(f"Efeito de forma pura (pareamento B, Uniforme-PERT): {efeito_forma_B:.2f} p.p.")
print(f"Parcela de momentos: {(efeito_total_A-efeito_forma_B)/efeito_total_A*100:.1f}%")
print(f"Parcela de forma pura: {efeito_forma_B/efeito_total_A*100:.1f}%")


Alvo de momentos (media, variancia exatas da PERT canonica):
  I: mu=503333.3  var=122222222.2
  R: mu=301000.0  var=155000000.0
  C: mu=150500.0  var=25892857.1

Família      E[VPL] R$mil  DP R$mil  P(VPL<0) +-SE(pp)
Triangular           39.3      49.8    22.59%    0.13%
PERT                 39.3      49.8    22.93%    0.13%
Uniforme             39.3      49.8    24.87%    0.14%
Normal eq.           39.3      49.8    21.52%    0.13%

Delta vs. PERT (pontos percentuais) -- efeito PURO de forma:
  Triangular   -0.34 p.p.
  PERT         +0.00 p.p.
  Uniforme     +1.94 p.p.
  Normal eq.   -1.41 p.p.

Efeito total (pareamento A, Uniforme-PERT): 11.77 p.p.
Efeito de forma pura (pareamento B, Uniforme-PERT): 1.94 p.p.
Parcela de momentos: 83.5%
Parcela de forma pura: 16.5%


In [9]:
"""
Eixo 2 — magnitude da incerteza (CV).

Decisão de desenho (documentada no artigo como resolução do \\pendente):
elicitação SIMÉTRICA em torno da moda nominal (mesma dos casos canônicos),
com a semiamplitude L calibrada para atingir cada CV-alvo via a fórmula
fechada do Corolário "Caso simétrico" (sigma^2_PERT = L^2/28):

    L = sigma_alvo * sqrt(28) ,  sigma_alvo = CV * mu_nominal
    a = m - L/2 ,  b = m + L/2

Isola "quanto de incerteza" sem confundir com a assimetria do caso canônico.
Família fixa: PERT (ver justificativa no código principal).
"""
import numpy as np
from scipy.stats import norm

CVS = [0.05, 0.10, 0.15, 0.20, 0.25]
NOMES_NOMINAIS = {k: v["m"] for k, v in CANON.items()}  # modas fixas


def elicitacao_simetrica(cv):
    L_FACTOR = np.sqrt(28)  # PERT simétrica: sigma^2 = L^2/28
    elicit = {}
    for var, m in NOMES_NOMINAIS.items():
        sigma_alvo = cv * m
        L = sigma_alvo * L_FACTOR
        elicit[var] = dict(a=m - L / 2, m=m, b=m + L / 2)
    return elicit


def rodar(N=N_DEFAULT, seed=MASTER_SEED):
    linhas = []
    for cv in CVS:
        elicit = elicitacao_simetrica(cv)
        rng = np.random.default_rng(seed)
        I, R, C = gerar_variaveis(rng, elicit, "PERT", N, rho=0.0)
        vpl = calcular_vpl(I, R, C, TMA_BASE)
        r = resumo(vpl, N)
        mu_exato, var_exato = momentos_exatos_vpl(elicit, "pert", TMA_BASE, rho=0.0)
        se_media = r["dp"] / np.sqrt(N)
        z_media = (r["media"] - mu_exato) / se_media
        linhas.append(dict(cv=cv, media=r["media"], dp=r["dp"], p_neg=r["p_neg"],
                            se_p=r["se_p"], p5=r["p5"], p95=r["p95"],
                            mu_exato=mu_exato, z_media=z_media))
    return linhas


if __name__ == "__main__":
    linhas = rodar()
    print(f"{'CV':>6}{'E[VPL] R$mil':>14}{'DP R$mil':>10}{'P(VPL<0)':>10}"
          f"{'+-SE(pp)':>9}{'P5 R$mil':>10}{'P95 R$mil':>10}{'z(media)':>9}")
    for l in linhas:
        print(f"{l['cv']*100:>5.0f}%"
              f"{l['media']/1000:>14.1f}"
              f"{l['dp']/1000:>10.1f}"
              f"{l['p_neg']*100:>9.2f}%"
              f"{l['se_p']*100:>8.2f}%"
              f"{l['p5']/1000:>10.1f}"
              f"{l['p95']/1000:>10.1f}"
              f"{l['z_media']:>9.2f}")

    # elasticidade aproximada do risco ao CV (log-log entre pontos extremos)
    p0, p1 = linhas[0]["p_neg"], linhas[-1]["p_neg"]
    cv0, cv1 = linhas[0]["cv"], linhas[-1]["cv"]
    elastic = (np.log(p1) - np.log(p0)) / (np.log(cv1) - np.log(cv0))
    print(f"\nElasticidade aproximada de P(VPL<0) ao CV "
          f"(entre {cv0*100:.0f}% e {cv1*100:.0f}%): {elastic:.2f}")

    CV  E[VPL] R$mil  DP R$mil  P(VPL<0) +-SE(pp)  P5 R$mil P95 R$mil z(media)
    5%          40.9      65.7    27.71%    0.14%     -66.8     149.1     0.76
   10%          41.0     131.3    38.44%    0.15%    -174.4     257.4     0.76
   15%          41.2     197.0    42.30%    0.16%    -281.9     365.8     0.76
   20%          41.3     262.6    44.15%    0.16%    -389.4     474.2     0.76
   25%          41.5     328.3    45.36%    0.16%    -497.0     582.5     0.76

Elasticidade aproximada de P(VPL<0) ao CV (entre 5% e 25%): 0.31


In [10]:
"""Eixo 3 — correlação rho(R,C), via cópula gaussiana. Elicitação canônica
(assimétrica, Tabela 'Projeto canônico'), família PERT, TMA base."""
import numpy as np
from scipy.stats import norm

RHOS = [-0.3, 0.0, 0.3, 0.6]


def rodar(N=N_DEFAULT, seed=MASTER_SEED):
    linhas = []
    for rho in RHOS:
        rng = np.random.default_rng(seed)  # mesmos Z1 base -> comparável
        I, R, C = gerar_variaveis(rng, CANON, "PERT", N, rho=rho)
        vpl = calcular_vpl(I, R, C, TMA_BASE)
        r = resumo(vpl, N)
        mu_exato, var_exato = momentos_exatos_vpl(CANON, "pert", TMA_BASE, rho=rho)
        se_media = r["dp"] / np.sqrt(N)
        z_media = (r["media"] - mu_exato) / se_media
        rho_emp = np.corrcoef(R, C)[0, 1]
        linhas.append(dict(rho=rho, rho_emp=rho_emp, media=r["media"], dp=r["dp"],
                            p_neg=r["p_neg"], se_p=r["se_p"], z_media=z_media,
                            var_exato=var_exato))
    return linhas


if __name__ == "__main__":
    linhas = rodar()
    base = linhas[1]["p_neg"]  # rho=0 como referência
    print(f"{'rho alvo':>9}{'rho emp.':>10}{'E[VPL] R$mil':>14}{'DP R$mil':>10}"
          f"{'P(VPL<0)':>10}{'+-SE(pp)':>9}{'vs rho=0 (pp)':>15}{'z(media)':>9}")
    for l in linhas:
        diff = (l["p_neg"] - base) * 100
        print(f"{l['rho']:>+9.1f}"
              f"{l['rho_emp']:>+10.3f}"
              f"{l['media']/1000:>14.1f}"
              f"{l['dp']/1000:>10.1f}"
              f"{l['p_neg']*100:>9.2f}%"
              f"{l['se_p']*100:>8.2f}%"
              f"{diff:>+14.1f}"
              f"{l['z_media']:>9.2f}")
    print("\nSinal previsto pela Proposição 'Momentos do VPL': "
          "rho>0 reduz Var[VPL] (hedge natural); confirmado acima "
          "pela monotonicidade decrescente de DP e P(VPL<0) em rho.")

 rho alvo  rho emp.  E[VPL] R$mil  DP R$mil  P(VPL<0) +-SE(pp)  vs rho=0 (pp) z(media)
     -0.3    -0.301          39.3      54.5    25.21%    0.14%          +2.2     0.50
     +0.0    -0.005          39.3      49.8    23.03%    0.13%          +0.0     0.51
     +0.3    +0.293          39.3      44.6    20.39%    0.13%          -2.6     0.53
     +0.6    +0.594          39.3      38.7    16.83%    0.12%          -6.2     0.60

Sinal previsto pela Proposição 'Momentos do VPL': rho>0 reduz Var[VPL] (hedge natural); confirmado acima pela monotonicidade decrescente de DP e P(VPL<0) em rho.


In [11]:
"""
Eixo 4 — TMA, via a identidade da Proposição 'Equivalência dos critérios':
    P(VPL(i) < 0) = F_TIR(i)   para todo i > -1.

Uma ÚNICA amostra de (I,R,C) (elicitação canônica, PERT, rho=0) é reaproveitada
para toda a curva: em vez de rodar uma simulação nova por nível de TMA,
avalia-se o VPL vetorizado numa grade de taxas -- exatamente a "consequência
computacional" descrita no artigo.
"""
import numpy as np
from scipy.stats import norm

TMAS_RELATORIO = [0.08, 0.10, 0.12, 0.15, 0.20]
GRADE_FINA = np.linspace(0.001, 0.40, 400)  # curva completa F_TIR


def rodar(N=N_DEFAULT, seed=MASTER_SEED):
    rng = np.random.default_rng(seed)
    I, R, C = gerar_variaveis(rng, CANON, "PERT", N, rho=0.0)

    def f_tir(i_grid):
        # vetoriza sobre a grade de taxas: VPL(i) para cada i, para toda a
        # amostra (I,R,C) de uma vez (broadcasting N x len(i_grid))
        S = fator_anuidade(i_grid)                      # (K,)
        vpl_grid = -I[:, None] + (R - C)[:, None] * S[None, :]  # (N,K)
        return (vpl_grid < 0).mean(axis=0)               # (K,)

    p_relatorio = f_tir(np.array(TMAS_RELATORIO))
    curva_completa = f_tir(GRADE_FINA)
    se = np.sqrt(p_relatorio * (1 - p_relatorio) / N)
    return TMAS_RELATORIO, p_relatorio, se, GRADE_FINA, curva_completa


if __name__ == "__main__":
    tmas, p_rel, se, grade, curva = rodar()
    print(f"{'TMA':>6}{'P(VPL<0)=F_TIR(TMA)':>22}{'+-SE(pp)':>10}")
    for tma, p, s in zip(tmas, p_rel, se):
        print(f"{tma*100:>5.0f}%{p*100:>21.2f}%{s*100:>9.2f}%")

    # TIR mediana (onde a curva cruza 50%) e alguns percentis, por interpolação
    p_alvo = [0.05, 0.25, 0.50, 0.75, 0.95]
    tir_percentis = np.interp(p_alvo, curva, grade)
    print("\nPercentis da TIR (via interpolação da curva F_TIR):")
    for p, t in zip(p_alvo, tir_percentis):
        print(f"  P{int(p*100):<3d} da TIR = {t*100:.2f}%")

    print(f"\nCaso base determinístico (Seção 'Projeto canônico'): TIR = 15,24%.")
    print("Confere com a leitura da curva: em TMA=15%, "
          f"P(VPL<0) = {np.interp(0.15, grade, curva)*100:.2f}% "
          "(TIR mediana simulada abaixo).")

   TMA   P(VPL<0)=F_TIR(TMA)  +-SE(pp)
    8%                 3.28%     0.06%
   10%                10.37%     0.10%
   12%                23.03%     0.13%
   15%                49.58%     0.16%
   20%                88.89%     0.10%

Percentis da TIR (via interpolação da curva F_TIR):
  P5   da TIR = 8.65%
  P25  da TIR = 12.26%
  P50  da TIR = 15.04%
  P75  da TIR = 17.86%
  P95  da TIR = 21.50%

Caso base determinístico (Seção 'Projeto canônico'): TIR = 15,24%.
Confere com a leitura da curva: em TMA=15%, P(VPL<0) = 49.58% (TIR mediana simulada abaixo).


In [12]:
"""Interações selecionadas (Seção 'Interações' do artigo).

(a) Família x CV: a ordem de risco entre famílias se mantém conforme a
    incerteza cresce, ou a interação inverte o ranking?
(b) Correlação x CV: o efeito-hedge da correlação positiva se amplifica ou
    se dilui quando a incerteza de base é maior?
"""
import numpy as np
from scipy.stats import norm

CVS_INTERACAO = [0.05, 0.15, 0.25]
FAMILIAS_INTERACAO = ["Triangular", "PERT", "Uniforme"]
RHOS_INTERACAO = [-0.3, 0.0, 0.6]


def familia_x_cv(N=N_DEFAULT, seed=MASTER_SEED):
    resultados = {}
    for familia in FAMILIAS_INTERACAO:
        linha = []
        for cv in CVS_INTERACAO:
            elicit = elicitacao_simetrica(cv)
            rng = np.random.default_rng(seed)
            I, R, C = gerar_variaveis(rng, elicit, familia, N, rho=0.0)
            vpl = calcular_vpl(I, R, C, TMA_BASE)
            r = resumo(vpl, N)
            linha.append(r["p_neg"])
        resultados[familia] = linha
    return resultados


def correlacao_x_cv(N=N_DEFAULT, seed=MASTER_SEED):
    resultados = {}
    for rho in RHOS_INTERACAO:
        linha = []
        for cv in CVS_INTERACAO:
            elicit = elicitacao_simetrica(cv)
            rng = np.random.default_rng(seed)
            I, R, C = gerar_variaveis(rng, elicit, "PERT", N, rho=rho)
            vpl = calcular_vpl(I, R, C, TMA_BASE)
            r = resumo(vpl, N)
            linha.append(r["p_neg"])
        resultados[rho] = linha
    return resultados


if __name__ == "__main__":
    print("=== Família x CV : P(VPL<0) ===")
    fam_cv = familia_x_cv()
    header = "Família".ljust(12) + "".join(f"CV={cv*100:.0f}%".rjust(10) for cv in CVS_INTERACAO)
    print(header)
    for familia, vals in fam_cv.items():
        print(familia.ljust(12) + "".join(f"{v*100:>9.2f}%" for v in vals))
    # amplitude do efeito-familia em cada CV (Uniforme - PERT)
    print("\nAmplitude (Uniforme - PERT), em pontos percentuais, por CV:")
    for j, cv in enumerate(CVS_INTERACAO):
        amp = (fam_cv["Uniforme"][j] - fam_cv["PERT"][j]) * 100
        print(f"  CV={cv*100:.0f}%: {amp:+.1f} p.p.")

    print("\n=== Correlação x CV : P(VPL<0) ===")
    header = "rho".ljust(8) + "".join(f"CV={cv*100:.0f}%".rjust(10) for cv in CVS_INTERACAO)
    print(header)
    corr_cv = correlacao_x_cv()
    for rho, vals in corr_cv.items():
        print(f"{rho:+.1f}".ljust(8) + "".join(f"{v*100:>9.2f}%" for v in vals))
    print("\nEfeito-hedge (rho=-0.3 menos rho=+0.6), em pontos percentuais, por CV:")
    for j, cv in enumerate(CVS_INTERACAO):
        efeito = (corr_cv[-0.3][j] - corr_cv[0.6][j]) * 100
        print(f"  CV={cv*100:.0f}%: {efeito:+.1f} p.p.")

=== Família x CV : P(VPL<0) ===
Família          CV=5%    CV=15%    CV=25%
Triangular      29.01%    42.75%    45.64%
PERT            27.71%    42.30%    45.36%
Uniforme        35.91%    45.30%    47.18%

Amplitude (Uniforme - PERT), em pontos percentuais, por CV:
  CV=5%: +8.2 p.p.
  CV=15%: +3.0 p.p.
  CV=25%: +1.8 p.p.

=== Correlação x CV : P(VPL<0) ===
rho          CV=5%    CV=15%    CV=25%
-0.3        29.68%    42.99%    45.73%
+0.0        27.71%    42.30%    45.36%
+0.6        21.96%    39.91%    43.92%

Efeito-hedge (rho=-0.3 menos rho=+0.6), em pontos percentuais, por CV:
  CV=5%: +7.7 p.p.
  CV=15%: +3.1 p.p.
  CV=25%: +1.8 p.p.


In [15]:
"""
VERSÃO ÚNICA PARA NOTEBOOK (Jupyter/Colab) — cole esta célula inteira e rode.
Não depende de nenhum outro arquivo/célula. Sem colisão de nomes entre eixos.

Gera as 5 figuras do capítulo de Resultados e salva em ./figuras/*.pdf
(e também mostra cada uma inline, com plt.show()).
"""
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats

# ===========================================================================
# 1. NÚCLEO (equivalente a mc_experiment.py)
# ===========================================================================
MASTER_SEED = 2026
N_DEFAULT = 100_000

CANON = {
    "I": dict(a=480_000, m=500_000, b=540_000),
    "R": dict(a=270_000, m=300_000, b=336_000),
    "C": dict(a=138_000, m=150_000, b=165_000),
}
N_ANOS = 5
TMA_BASE = 0.12


def fator_anuidade(tma, n=N_ANOS):
    tma = np.asarray(tma, dtype=float)
    return np.where(np.abs(tma) < 1e-12, float(n),
                     (1 - (1 + tma) ** (-n)) / tma)


def ppf_triangular(u, a, m, b):
    c = (m - a) / (b - a)
    return stats.triang.ppf(u, c=c, loc=a, scale=b - a)


def ppf_pert(u, a, m, b, lam=4.0):
    alpha1 = 1 + lam * (m - a) / (b - a)
    alpha2 = 1 + lam * (b - m) / (b - a)
    return a + (b - a) * stats.beta.ppf(u, alpha1, alpha2)


def ppf_uniforme(u, a, m, b):
    return stats.uniform.ppf(u, loc=a, scale=b - a)


def momentos_triangular(a, m, b):
    mu = (a + m + b) / 3
    var = (a**2 + m**2 + b**2 - a*m - a*b - m*b) / 18
    return mu, var


def momentos_pert(a, m, b, lam=4.0):
    mu = (a + lam * m + b) / (lam + 2)
    var = (mu - a) * (b - mu) / (lam + 3)
    return mu, var


def momentos_uniforme(a, m, b):
    mu = (a + b) / 2
    var = (b - a) ** 2 / 12
    return mu, var


MOMENTOS = {
    "triangular": momentos_triangular,
    "pert": momentos_pert,
    "uniforme": momentos_uniforme,
}


def ppf_normal_eq(u, a, m, b, ref="triangular"):
    mu, var = MOMENTOS[ref](a, m, b)
    return stats.norm.ppf(u, loc=mu, scale=np.sqrt(var))


FAMILIAS = {
    "Triangular": (ppf_triangular, "triangular"),
    "PERT": (ppf_pert, "pert"),
    "Uniforme": (ppf_uniforme, "uniforme"),
    "Normal eq.": (ppf_normal_eq, "triangular"),
}


def gerar_variaveis(rng, elicit, familia, N, rho=0.0):
    ppf_func, _ = FAMILIAS[familia]
    u_I = rng.random(N)
    I = ppf_func(u_I, **elicit["I"])
    z1 = rng.standard_normal(N)
    z2_indep = rng.standard_normal(N)
    z2 = rho * z1 + np.sqrt(1 - rho**2) * z2_indep
    u_R = stats.norm.cdf(z1)
    u_C = stats.norm.cdf(z2)
    R = ppf_func(u_R, **elicit["R"])
    C = ppf_func(u_C, **elicit["C"])
    return I, R, C


def calcular_vpl(I, R, C, tma, n=N_ANOS):
    S = fator_anuidade(tma, n)
    return -I + (R - C) * S


def resumo(vpl, N):
    media = vpl.mean()
    dp = vpl.std(ddof=1)
    p_neg = (vpl < 0).mean()
    se_p = np.sqrt(p_neg * (1 - p_neg) / N)
    p5, p50, p95 = np.percentile(vpl, [5, 50, 95])
    return dict(media=media, dp=dp, p_neg=p_neg, se_p=se_p, p5=p5, p50=p50, p95=p95)


def momentos_exatos_vpl(elicit, familia_key, tma, rho=0.0, n=N_ANOS):
    S = fator_anuidade(tma, n)
    mu_I, var_I = MOMENTOS[familia_key](**elicit["I"])
    mu_R, var_R = MOMENTOS[familia_key](**elicit["R"])
    mu_C, var_C = MOMENTOS[familia_key](**elicit["C"])
    sig_R, sig_C = np.sqrt(var_R), np.sqrt(var_C)
    media = -mu_I + (mu_R - mu_C) * S
    var = var_I + S**2 * (var_R + var_C - 2 * rho * sig_R * sig_C)
    return media, var


def elicitacao_simetrica(cv):
    L_FACTOR = np.sqrt(28)
    elicit = {}
    for var, ponto in CANON.items():
        m = ponto["m"]
        sigma_alvo = cv * m
        L = sigma_alvo * L_FACTOR
        elicit[var] = dict(a=m - L / 2, m=m, b=m + L / 2)
    return elicit


# ===========================================================================
# 2. CÁLCULOS DE CADA EIXO (nomes únicos -- sem colisão)
# ===========================================================================
def rodar_eixo2_cv(N=N_DEFAULT, seed=MASTER_SEED):
    linhas = []
    for cv in [0.05, 0.10, 0.15, 0.20, 0.25]:
        elicit = elicitacao_simetrica(cv)
        rng = np.random.default_rng(seed)
        I, R, C = gerar_variaveis(rng, elicit, "PERT", N, rho=0.0)
        vpl = calcular_vpl(I, R, C, TMA_BASE)
        r = resumo(vpl, N)
        linhas.append(dict(cv=cv, **r))
    return linhas


def rodar_eixo3_correlacao(N=N_DEFAULT, seed=MASTER_SEED):
    linhas = []
    for rho in [-0.3, 0.0, 0.3, 0.6]:
        rng = np.random.default_rng(seed)
        I, R, C = gerar_variaveis(rng, CANON, "PERT", N, rho=rho)
        vpl = calcular_vpl(I, R, C, TMA_BASE)
        r = resumo(vpl, N)
        linhas.append(dict(rho=rho, **r))
    return linhas


def rodar_eixo4_tma(N=N_DEFAULT, seed=MASTER_SEED):
    rng = np.random.default_rng(seed)
    I, R, C = gerar_variaveis(rng, CANON, "PERT", N, rho=0.0)
    tmas_relatorio = [0.08, 0.10, 0.12, 0.15, 0.20]
    grade_fina = np.linspace(0.001, 0.40, 400)

    def f_tir(i_grid):
        S = fator_anuidade(i_grid)
        vpl_grid = -I[:, None] + (R - C)[:, None] * S[None, :]
        return (vpl_grid < 0).mean(axis=0)

    p_relatorio = f_tir(np.array(tmas_relatorio))
    curva_completa = f_tir(grade_fina)
    return tmas_relatorio, p_relatorio, grade_fina, curva_completa


def rodar_familia_x_cv(N=N_DEFAULT, seed=MASTER_SEED):
    resultados = {}
    for familia in ["Triangular", "PERT", "Uniforme"]:
        linha = []
        for cv in [0.05, 0.15, 0.25]:
            elicit = elicitacao_simetrica(cv)
            rng = np.random.default_rng(seed)
            I, R, C = gerar_variaveis(rng, elicit, familia, N, rho=0.0)
            vpl = calcular_vpl(I, R, C, TMA_BASE)
            linha.append(resumo(vpl, N)["p_neg"])
        resultados[familia] = linha
    return resultados


def rodar_correlacao_x_cv(N=N_DEFAULT, seed=MASTER_SEED):
    resultados = {}
    for rho in [-0.3, 0.0, 0.6]:
        linha = []
        for cv in [0.05, 0.15, 0.25]:
            elicit = elicitacao_simetrica(cv)
            rng = np.random.default_rng(seed)
            I, R, C = gerar_variaveis(rng, elicit, "PERT", N, rho=rho)
            vpl = calcular_vpl(I, R, C, TMA_BASE)
            linha.append(resumo(vpl, N)["p_neg"])
        resultados[rho] = linha
    return resultados


# ===========================================================================
# 3. FIGURAS
# ===========================================================================
OUTDIR = "figuras"
os.makedirs(OUTDIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 150,
})
COR = {"Triangular": "#1f77b4", "PERT": "#d62728", "Uniforme": "#2ca02c", "Normal eq.": "#7f7f7f"}
MARC = {"Triangular": "o", "PERT": "s", "Uniforme": "^", "Normal eq.": "D"}
TRACO = {"Triangular": "-", "PERT": "-", "Uniforme": "-", "Normal eq.": "--"}


def fig_eixo1_densidades():
    fig, ax = plt.subplots(figsize=(6.3, 4.0))
    grade = np.linspace(-350_000, 550_000, 800)
    for nome, (_, ref_key) in FAMILIAS.items():
        rng = np.random.default_rng(MASTER_SEED)
        I, R, C = gerar_variaveis(rng, CANON, nome, N_DEFAULT, rho=0.0)
        vpl = calcular_vpl(I, R, C, TMA_BASE)
        kde = stats.gaussian_kde(vpl)
        ax.plot(grade / 1000, kde(grade) * 1000, label=nome,
                color=COR[nome], linestyle=TRACO[nome], linewidth=1.8)
    ax.axvline(0, color="black", linewidth=0.9, linestyle=":")
    ax.set_xlabel("VPL (R$ mil)")
    ax.set_ylabel("densidade")
    ax.set_xlim(-350, 550)
    ax.legend(frameon=False, loc="upper right", fontsize=9)
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig_eixo1_densidades.pdf")
    plt.show()


def fig_eixo2_cv():
    linhas = rodar_eixo2_cv()
    cvs = [l["cv"] * 100 for l in linhas]
    p_neg = [l["p_neg"] * 100 for l in linhas]
    dp = [l["dp"] / 1000 for l in linhas]
    fig, ax1 = plt.subplots(figsize=(6.3, 4.0))
    ax1.plot(cvs, p_neg, "o-", color="#d62728", linewidth=1.8)
    ax1.set_xlabel("Coeficiente de variação (CV, %)")
    ax1.set_ylabel(r"$\mathbb{P}(VPL<0)$ (%)", color="#d62728")
    ax1.tick_params(axis="y", labelcolor="#d62728")
    ax1.set_ylim(20, 50)
    ax2 = ax1.twinx()
    ax2.plot(cvs, dp, "s--", color="#1f77b4", linewidth=1.5)
    ax2.set_ylabel("Desvio-padrão do VPL (R$ mil)", color="#1f77b4")
    ax2.tick_params(axis="y", labelcolor="#1f77b4")
    ax2.grid(False)
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig_eixo2_cv.pdf")
    plt.show()


def fig_eixo3_correlacao():
    linhas = rodar_eixo3_correlacao()
    rhos = [l["rho"] for l in linhas]
    p_neg = [l["p_neg"] * 100 for l in linhas]
    fig, ax = plt.subplots(figsize=(6.3, 4.0))
    ax.plot(rhos, p_neg, "o-", color="#d62728", linewidth=1.8)
    ax.axhline(p_neg[rhos.index(0.0)], color="gray", linewidth=0.8, linestyle=":")
    ax.set_xlabel(r"Correlação $\rho(R,C)$")
    ax.set_ylabel(r"$\mathbb{P}(VPL<0)$ (%)")
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig_eixo3_correlacao.pdf")
    plt.show()


def fig_eixo4_curva_risco():
    tmas_rel, p_rel, grade, curva = rodar_eixo4_tma()
    fig, ax = plt.subplots(figsize=(6.3, 4.0))
    ax.plot(grade * 100, curva * 100, "-", color="#d62728", linewidth=1.8)
    ax.plot(np.array(tmas_rel) * 100, np.array(p_rel) * 100, "o",
            color="#d62728", markersize=6, zorder=5)
    for tma, p in zip(tmas_rel, p_rel):
        ax.annotate(f"{p*100:.1f}%", xy=(tma*100, p*100),
                    xytext=(tma*100+0.4, p*100+2), fontsize=8)
    ax.axhline(50, color="gray", linewidth=0.8, linestyle=":")
    ax.axvline(15.24, color="black", linewidth=0.8, linestyle="--")
    ax.text(15.24 + 0.4, 5, "TIR determinística\n(caso base) = 15,24%", fontsize=8)
    ax.set_xlabel("Taxa (TMA / i, % a.a.)")
    ax.set_ylabel(r"$\mathbb{P}(VPL<0)$ (%)")
    ax.set_xlim(0, 30)
    ax.set_ylim(0, 100)
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig_eixo4_curva_risco.pdf")
    plt.show()


def fig_interacoes():
    fam_cv = rodar_familia_x_cv()
    corr_cv = rodar_correlacao_x_cv()
    cvs = [cv * 100 for cv in [0.05, 0.15, 0.25]]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.2, 4.0), sharey=True)
    for familia, vals in fam_cv.items():
        ax1.plot(cvs, [v * 100 for v in vals], marker=MARC[familia],
                  color=COR[familia], linestyle=TRACO[familia],
                  linewidth=1.6, label=familia)
    ax1.set_xlabel("CV (%)")
    ax1.set_ylabel(r"$\mathbb{P}(VPL<0)$ (%)")
    ax1.set_title("(a) Família x CV", fontsize=10)
    ax1.legend(frameon=False, fontsize=8.5, loc="lower right")
    cores_rho = {-0.3: "#d62728", 0.0: "#7f7f7f", 0.6: "#1f77b4"}
    for rho, vals in corr_cv.items():
        ax2.plot(cvs, [v * 100 for v in vals], marker="o",
                  color=cores_rho[rho], linewidth=1.6, label=fr"$\rho={rho:+.1f}$")
    ax2.set_xlabel("CV (%)")
    ax2.set_title(r"(b) Correlação x CV", fontsize=10)
    ax2.legend(frameon=False, fontsize=8.5, loc="lower right")
    fig.tight_layout()
    fig.savefig(f"{OUTDIR}/fig_interacoes.pdf")
    plt.show()


# ===========================================================================
# 4. RODAR TUDO
# ===========================================================================
fig_eixo1_densidades()
fig_eixo2_cv()
fig_eixo3_correlacao()
fig_eixo4_curva_risco()
fig_interacoes()
print("Figuras salvas em", os.path.abspath(OUTDIR)) 

C:\Users\bruna\AppData\Local\Temp\ipykernel_8208\2914588757.py:242: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\bruna\AppData\Local\Temp\ipykernel_8208\2914588757.py:263: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\bruna\AppData\Local\Temp\ipykernel_8208\2914588757.py:277: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\bruna\AppData\Local\Temp\ipykernel_8208\2914588757.py:298: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figuras salvas em C:\Users\bruna\MEGA\Jupyter Notebooks\figuras


C:\Users\bruna\AppData\Local\Temp\ipykernel_8208\2914588757.py:323: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
